# Phase 6 — Deep Learning (OPTIONAL)

Thin wrapper around `src/train_dl.py` (sliding-window LSTM). This phase is optional — skip this notebook entirely if your timeline doesn't have room for it; Phases 1-5 and 7-9 don't depend on it.

Requires `pip install -r requirements-dl.txt` (TensorFlow).

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from src import config, evaluate
from src.train_dl import build_windows, split_by_target_date, WINDOW_SIZE, _require_tensorflow

In [ ]:
tf = _require_tensorflow()
from tensorflow import keras

In [ ]:
MODE = 'total'

clean_df = pd.read_csv(config.CLEAN_CSV_PATH, parse_dates=[config.DATE_COL])
series_df = (clean_df[clean_df[config.MODE_COL] == MODE]
             .sort_values(config.DATE_COL)[[config.DATE_COL, config.TARGET_COL]]
             .dropna().reset_index(drop=True))

X, y, target_dates = build_windows(series_df, window=WINDOW_SIZE)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_by_target_date(X, y, target_dates)
X_train.shape, X_val.shape, X_test.shape

## Standardize using TRAIN stats only, then build + train the LSTM

In [ ]:
mean, std = y_train.mean(), y_train.std()
X_train_s, X_val_s, X_test_s = (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std
y_train_s, y_val_s = (y_train - mean) / std, (y_val - mean) / std

keras.utils.set_random_seed(config.RANDOM_SEED)
model = keras.Sequential([
    keras.layers.Input(shape=(WINDOW_SIZE, 1)),
    keras.layers.LSTM(64),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1),
])
model.compile(optimizer='adam', loss='mse')
history = model.fit(
    X_train_s, y_train_s, validation_data=(X_val_s, y_val_s),
    epochs=100, batch_size=32, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend(); plt.title('LSTM training curve (standardized MSE)'); plt.show()

## Score on the test window (back-transformed to raw ridership units)

In [ ]:
y_pred_s = model.predict(X_test_s, verbose=0).flatten()
y_pred = y_pred_s * std + mean
evaluate.score_all(y_test, y_pred)

## Verdict vs. Phase 5's XGBoost

Run `python -m src.train_dl` for the full loop over all three modes plus the printed trade-off verdict against `reports/ml_metrics.csv` — reuse that logic here if you want it inline:

In [ ]:
from src.train_dl import _print_verdict_vs_xgboost
_print_verdict_vs_xgboost(MODE, evaluate.rmse(y_test, y_pred))